In [1]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from src.preprocess import preprocess_data

In [4]:
print("Loading and preprocessing training data...")
train_data = pd.read_csv(r'../data/raw/train.csv')
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

Loading and preprocessing training data...


In [5]:
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
print("Applying SMOTE....")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_local, y_train_local)

local_model = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
print("Training local model...")
local_model.fit(X_train_smote, y_train_smote)
local_preds = local_model.predict(X_test_local)
macro_f1 = f1_score(y_test_local, local_preds, average='macro')

print(f"\nLocal Validation Macro F1-Score: {macro_f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test_local, local_preds))

Applying SMOTE....
Training local model...

Local Validation Macro F1-Score: 0.8215

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.94      0.95    118512
           1       0.76      0.82      0.79      7961
           2       0.65      0.82      0.72     11545

    accuracy                           0.92    138018
   macro avg       0.79      0.86      0.82    138018
weighted avg       0.93      0.92      0.93    138018



In [7]:
print("Applying SMOTE to ALL training data...")
X_full_smote, y_full_smote = smote.fit_resample(X, y)

Applying SMOTE to ALL training data...


In [9]:
# 2. Retrain the model on 100% of the SMOTE Data
final_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss')
print("Retraining final model...")
final_model.fit(X_full_smote, y_full_smote)

# 3. Load and preprocess the Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_4.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Retraining final model...
Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


In [ ]:
import optuna